In [1]:
import pandas as pd
import polars as pl
import numpy as np
from tqdm.auto import tqdm
from matplotlib import pyplot as plt
import pickle
import warnings

warnings.filterwarnings("ignore")
pd.options.display.max_columns = None
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(400)
pl.Config.set_fmt_table_cell_list_len(5)

polars.config.Config

In [2]:
class CONFIG:
    seed = 42
    target_col = "responder_6"
    lag_cols_rename = {
        f"responder_{idx}_lag_1": f"responder_{idx}" for idx in range(9)
    }
    lag_target_cols_name = [f"responder_{idx}" for idx in range(9)]
    lag_cols_original = ["date_id", "time_id", "symbol_id"] + [
        f"responder_{idx}" for idx in range(9)
    ]
    model_path = "./input/janestreet-public-model/xgb_001.pkl"
    lag_ndays = 4

In [3]:
def create_agg_list(day, columns):
    agg_mean_list = [
        pl.col(c).mean().name.suffix(f"_mean_{day}d") for c in columns
    ]
    agg_std_list = [
        pl.col(c).std().name.suffix(f"_std_{day}d") for c in columns
    ]
    agg_max_list = [
        pl.col(c).max().name.suffix(f"_max_{day}d") for c in columns
    ]
    agg_last_list = [
        pl.col(c).last().name.suffix(f"_last_{day}d") for c in columns
    ]
    agg_list = agg_mean_list + agg_std_list + agg_max_list + agg_last_list
    return agg_list

In [4]:
train = pl.read_parquet(
    "./input/jane-street-real-time-market-data-forecasting/train.parquet"
)
lags = train.select(pl.col(CONFIG.lag_cols_original))
lags = lags.with_columns(date_id=pl.col("date_id") + 1)
date_ids = lags.select("date_id").unique().to_series()
agg_list = create_agg_list(1, CONFIG.lag_target_cols_name)
result = []
for date_id in tqdm(date_ids, total=len(date_ids)):
    try:
        lags_ = lags.filter(
            (pl.col("date_id") > date_id - CONFIG.lag_ndays)
            & (pl.col("date_id") <= date_id)
        )
        lags_ = lags.filter((pl.col("date_id") == date_id - CONFIG.lag_ndays))
        lags_ = lags_.with_columns(date_id=date_id)
        lags_ = lags_.group_by(
            ["date_id", "symbol_id"], maintain_order=True
        ).agg(agg_list)
        result.append(lags_)
    except:
        continue
lag_Ndays = pl.concat(result).sort("date_id")
lag_Ndays = lag_Ndays.cast({"date_id": pl.Int16})
train = train.join(lag_Ndays, on=["date_id", "symbol_id"], how="left")

  0%|          | 0/1699 [00:00<?, ?it/s]

In [5]:
with open(CONFIG.model_path, "rb") as fp:
    result = pickle.load(fp)
model = result["model"]
features = result["features"]

In [6]:
history = (
    pl.scan_parquet(
        "./input/jane-street-real-time-market-data-forecasting/train.parquet"
    )
    .select(
        ["date_id", "time_id", "symbol_id"]
        + [f"responder_{idx}" for idx in range(9)]
    )
    .filter(
        (pl.col("date_id") >= (1698 - CONFIG.lag_ndays))
        & (pl.col("date_id") < 1698)
    )
)
history = history.with_columns(
    date_id=(pl.col("date_id") - pl.lit(1698)).cast(pl.Int16)
)
history = history.collect()
history_column_types = {
    "date_id": pl.Int16,
    "time_id": pl.Int16,
    "symbol_id": pl.Int16,
}
feature_column_types = {}
for f in [f"feature_{idx:02d}" for idx in range(79)]:
    feature_column_types[f] = pl.Float32
responder_column_types = {}
for f in [f"responder_{idx}" for idx in range(9)]:
    responder_column_types[f] = pl.Float32
history = history.cast(history_column_types)
history = history.cast(responder_column_types)
history.tail()

date_id,time_id,symbol_id,responder_0,responder_1,responder_2,responder_3,responder_4,responder_5,responder_6,responder_7,responder_8
i16,i16,i16,f32,f32,f32,f32,f32,f32,f32,f32,f32
-1,967,34,0.501321,0.905332,-0.819582,-0.564046,-0.223018,-0.283954,-0.045938,0.009797,-0.102538
-1,967,35,-1.113053,0.69719,-1.619031,-1.222743,-0.706082,-0.291133,0.167733,0.099704,0.32461
-1,967,36,-1.019353,-0.460962,-2.026678,-0.848606,-0.305448,-1.256913,-0.109359,-0.027474,-0.253956
-1,967,37,0.23585,0.556479,0.618944,-0.243765,-0.108361,-0.260777,-0.486923,-0.275566,-1.020708
-1,967,38,0.542563,0.513193,0.814393,0.032767,0.025435,0.311465,-0.044797,0.011133,-0.0793


In [7]:
def predict(
    test: pl.DataFrame, lags: pl.DataFrame | None
) -> pl.DataFrame | pd.DataFrame:
    global history
    global lags_infer
    current_date = test.select("date_id").to_numpy()[:, 0][0]
    if lags is not None:
        lags = lags.rename(CONFIG.lag_cols_rename)
        lags = lags.cast(history_column_types)
        lags = lags.cast(responder_column_types)
        history = pl.concat([history, lags])
        history = history.filter(
            pl.col("date_id") > (current_date - CONFIG.lag_ndays)
        )
        agg_list = create_agg_list(1, CONFIG.lag_target_cols_name)
        shift_n_data = history.filter(pl.col("date_id") == current_date)
        lags_infer = shift_n_data.group_by(
            ["date_id", "symbol_id"], maintain_order=True
        ).agg(agg_list)
    test = test.cast(history_column_types)
    test = test.cast(feature_column_types)
    X_test = test.join(lags_infer, on=["date_id", "symbol_id"], how="left")
    preds = np.zeros((X_test.shape[0],))
    preds += model.predict(X_test[features].to_pandas().values)
    preds = np.clip(preds, a_min=-5, a_max=5)
    predictions = test.select("row_id").with_columns(
        pl.Series(name="responder_6", values=preds, dtype=pl.Float64)
    )
    assert isinstance(predictions, pl.DataFrame | pd.DataFrame)
    assert list(predictions.columns) == ["row_id", "responder_6"]
    assert len(predictions) == len(test)
    return predictions

In [8]:
test = pl.read_parquet(
    "./input/jane-street-real-time-market-data-forecasting/test.parquet"
)
lags = pl.read_parquet(
    "./input/jane-street-real-time-market-data-forecasting/lags.parquet"
)
predict(test, lags)

row_id,responder_6
i64,f64
0,0.276523
1,0.295364
2,0.000829
3,0.043334
4,0.307003
5,0.19524
6,0.085774
7,0.320546
8,0.23595
